# ACT(基準線)の学習 — 積み木のお片付け
Google Drive の同期フォルダ(`PAI2026 Workspace`。自動で探す)にあるデータで ACT を学習し、チェックポイントを同じ Drive フォルダに保存する。
**ランタイム → ランタイムのタイプを変更 → GPU(T4 など)** にしてから、上から順に実行する。

Drive の同期フォルダには、次の4つがある想定:
- `lerobot_dataset/s1_region`(変換済みデータ)  - `pai_tools/tools/pai_img/train_img.py`(学習コード)
- `lerobot_src`(ローカルで使っている lerobot のソース。`docker cp` で出したもの)  - `checkpoints`(結果の置き場)

In [ ]:
# 1) Drive をつなぎ、同期フォルダを自動で探す(lerobot_dataset を含むフォルダ)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sys, shutil, subprocess
def sh(cmd):
    # コマンドを実行して出力をそのまま表示する。失敗したら止める
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()
    assert p.returncode == 0, '失敗(終了コード %d): %s' % (p.returncode, cmd)
NAME = 's1_region'                                   # データ名(= チェックポイント名)
DRIVE_ROOT = None                                    # 自動で見つからないときだけ、ここに '/content/drive/MyDrive/…' を直接書く
if DRIVE_ROOT is None:
    hits = sorted(glob.glob('/content/drive/MyDrive/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/*/lerobot_dataset/'+NAME))
    print('見つかった候補:', hits)
    assert hits, 'lerobot_dataset/%s が Drive に見つからない。sync_out.bat の同期が終わっているか確認し、DRIVE_ROOT を直接書く' % NAME
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print('DRIVE_ROOT =', DRIVE_ROOT)
for p in ['lerobot_dataset/'+NAME, 'pai_tools/tools/pai_img/train_img.py', 'lerobot_src/pyproject.toml']:
    print(('OK  ' if os.path.exists(f'{DRIVE_ROOT}/{p}') else '無い ') + p)

In [ ]:
# 2) GPU の確認
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
print(sys.version)

In [ ]:
# 3) lerobot と、ローカルに合わせた版の依存を入れる(数分)
#    ※ 先に「ランタイム → セッションを再起動」しておくこと(torch を入れ替えるため)
shutil.rmtree('/content/lerobot_src', ignore_errors=True)
shutil.copytree(f'{DRIVE_ROOT}/lerobot_src', '/content/lerobot_src')
sh('pip install -q torch==2.7.0 torchvision==0.22.0 torchcodec==0.4.0')
sh('pip install -q --only-binary=:all: "av>=14.2.0" datasets==3.6.0')
sh('pip install -q -e /content/lerobot_src')
sys.path.insert(0, '/content/lerobot_src')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
from lerobot.common.policies.act.modeling_act import ACTPolicy; print('lerobot import OK')

In [ ]:
# 4) データを Colab のローカルディスクへ(Drive のまま読むと遅い)
shutil.rmtree('/content/dataset', ignore_errors=True)
os.makedirs('/content/dataset')
shutil.copytree(f'{DRIVE_ROOT}/lerobot_dataset/{NAME}', f'/content/dataset/{NAME}')
shutil.copy(f'{DRIVE_ROOT}/pai_tools/tools/pai_img/train_img.py', '/content/train_img.py')
print(os.listdir(f'/content/dataset/{NAME}'))

In [ ]:
# 5) まず短く動作確認(30ステップ、1分以内)
sh(f'python /content/train_img.py --dataset-root /content/dataset/{NAME} --repo-id {NAME} --output /content/ckpt_test --steps 30 --batch-size 8 --num-workers 2')

In [ ]:
# 6) 本番の学習(80000ステップ、L4で約1.5時間)。30000ステップの結果は消さないよう、別フォルダに保存する
#    10000ステップごとに Drive へ保存する(途中で切れても、そこまでは残る)
STEPS = 80000
OUT = f'{DRIVE_ROOT}/checkpoints/train/{NAME}_80k'
sh(f'python /content/train_img.py --dataset-root /content/dataset/{NAME} --repo-id {NAME} --output "{OUT}" --steps {STEPS} --batch-size 8 --num-workers 2 --save-every 10000')


In [ ]:
# 7) 保存の確認(config.json と model.safetensors があればよい)
sh(f'ls -la "{OUT}"')

## 終わったら
PC 側(Gazebo のある端末)で `sync_in.bat` を実行すると、Drive の `checkpoints` がコンテナに入る。
そのあと `infer_act.bat s1_region`(Gazebo 起動後)で、ロボットを動かして評価する。

注意: Colab は、ブラウザのタブを閉じる、無操作が続く、使用時間の上限、のどれかで切れる。学習コードに再開機能はないので、途中で切れたら、最後に保存された 5000 ステップ単位のチェックポイントまでが残る。